# LESSON 4.5: Image Sharpening Using Highpass Filters
## Filtering in the Frequency Domain

In this lesson:
- Highpass filters: attenuate low frequencies, pass high frequencies
- Relationship between lowpass and highpass filters
- Ideal Highpass Filter (IHPF)
- Gaussian Highpass Filter (GHPF)
- Butterworth Highpass Filter (BHPF)
- Comparison of all three highpass filter types
- High-frequency emphasis filtering
- Laplacian in the frequency domain

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Introduction: Highpass Filters

In the frequency domain, images consist of a range of frequency components:
- **Low frequencies** correspond to slowly varying regions (smooth areas, background)
- **High frequencies** correspond to rapid intensity changes (edges, fine detail, noise)

A **highpass filter** attenuates low-frequency components while passing (or boosting) high-frequency components. This is the fundamental operation for **image sharpening** and **edge detection** in the frequency domain.

### The Filtering Process (Gonzalez, Chapter 4)

The general steps for frequency domain filtering:

1. Multiply the input image $f(x,y)$ by $(-1)^{x+y}$ to center the transform
2. Compute the 2-D DFT: $F(u,v) = \mathcal{F}\{f(x,y)\}$
3. Multiply by the filter transfer function: $G(u,v) = H(u,v) \cdot F(u,v)$
4. Compute the inverse DFT and take the real part
5. Multiply by $(-1)^{x+y}$ to undo the centering

In NumPy, steps 1 and 5 are handled by `fftshift` / `ifftshift`.

In [ ]:
# Helper function: create a synthetic biomedical-like test image
def create_test_image(size=256):
    """Create a synthetic test image with edges, smooth regions, and fine detail."""
    img = np.zeros((size, size), dtype=np.float64)
    
    # Smooth background gradient
    Y, X = np.mgrid[0:size, 0:size]
    img += 50 + 30 * np.sin(2 * np.pi * X / size) * np.cos(2 * np.pi * Y / size)
    
    # Large elliptical structure (simulating an organ or tissue boundary)
    cy, cx = size // 2, size // 2
    ellipse = ((X - cx) / 70)**2 + ((Y - cy) / 50)**2
    img[ellipse <= 1] = 180
    
    # Smaller circular structures inside (simulating lesions or vessels)
    for (dx, dy, r, val) in [(-25, -15, 12, 220), (20, 10, 10, 60), (0, 25, 8, 240)]:
        dist = np.sqrt((X - cx - dx)**2 + (Y - cy - dy)**2)
        img[dist <= r] = val
    
    # Fine horizontal and vertical lines (simulating fine structures)
    img[size//4, size//4:3*size//4] = 255
    img[size//4:3*size//4, 3*size//4] = 255
    
    # Small bright dots (simulating point features)
    np.random.seed(42)
    for _ in range(15):
        px, py = np.random.randint(30, size-30, 2)
        img[py-1:py+2, px-1:px+2] = 255
    
    return np.clip(img, 0, 255)


# Helper function: distance matrix from center
def distance_from_center(shape):
    """Compute D(u,v) = distance from center of frequency rectangle."""
    M, N = shape
    u = np.arange(M) - M // 2
    v = np.arange(N) - N // 2
    V, U = np.meshgrid(v, u)
    D = np.sqrt(U**2 + V**2)
    return D


# Helper function: apply frequency domain filter
def apply_filter(image, H):
    """Apply a frequency domain filter H(u,v) to an image."""
    F = np.fft.fftshift(np.fft.fft2(image))
    G = H * F
    g = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
    return g


# Create and display the test image
test_img = create_test_image(256)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Test Image f(x,y)', fontsize=12)
axes[0].axis('off')

F = np.fft.fftshift(np.fft.fft2(test_img))
axes[1].imshow(np.log1p(np.abs(F)), cmap='gray')
axes[1].set_title('Centered Spectrum |F(u,v)| (log scale)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Synthetic Test Image for Highpass Filtering', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Relationship Between Lowpass and Highpass Filters

A highpass filter can be obtained from a corresponding lowpass filter by a simple relationship:

$$\boxed{H_{HP}(u,v) = 1 - H_{LP}(u,v)}$$

Where:
- $H_{HP}(u,v)$ is the highpass filter transfer function
- $H_{LP}(u,v)$ is the corresponding lowpass filter transfer function

### Intuition:
- A lowpass filter passes low frequencies and blocks high frequencies
- Subtracting the lowpass from 1 inverts this behavior: blocks low, passes high
- This means: **highpass filtered image = original - lowpass filtered image**

In the spatial domain, this corresponds to:

$$g_{HP}(x,y) = f(x,y) - g_{LP}(x,y)$$

This is the principle behind **unsharp masking** (subtracting a blurred version from the original).

In [ ]:
# Demonstrate the relationship H_HP = 1 - H_LP
D = distance_from_center(test_img.shape)
D0 = 30  # cutoff frequency

# Gaussian lowpass
H_LP = np.exp(-D**2 / (2 * D0**2))

# Highpass from lowpass
H_HP = 1 - H_LP

# Apply both filters
g_lp = apply_filter(test_img, H_LP)
g_hp = apply_filter(test_img, H_HP)

# Verify: original - lowpass = highpass
g_diff = test_img - g_lp

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Filter transfer functions
axes[0, 0].imshow(H_LP, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('$H_{LP}(u,v)$ (Gaussian LP)', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(H_HP, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('$H_{HP}(u,v) = 1 - H_{LP}(u,v)$', fontsize=12)
axes[0, 1].axis('off')

# Radial cross section
center = test_img.shape[0] // 2
radial_lp = H_LP[center, center:]
radial_hp = H_HP[center, center:]
freqs = np.arange(len(radial_lp))
axes[0, 2].plot(freqs, radial_lp, 'b-', linewidth=2, label='$H_{LP}$')
axes[0, 2].plot(freqs, radial_hp, 'r-', linewidth=2, label='$H_{HP} = 1 - H_{LP}$')
axes[0, 2].axvline(x=D0, color='gray', linestyle='--', label=f'$D_0 = {D0}$')
axes[0, 2].set_xlabel('Distance from center D(u,v)', fontsize=11)
axes[0, 2].set_ylabel('H(u,v)', fontsize=11)
axes[0, 2].set_title('Radial Cross Section', fontsize=12)
axes[0, 2].legend(fontsize=10)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].set_ylim([-0.05, 1.05])

# Filtered results
axes[1, 0].imshow(g_lp, cmap='gray')
axes[1, 0].set_title('Lowpass Filtered $g_{LP}$', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(g_hp, cmap='gray')
axes[1, 1].set_title('Highpass Filtered $g_{HP}$', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(g_diff, cmap='gray')
axes[1, 2].set_title('$f - g_{LP}$ (same as $g_{HP}$)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Relationship: $H_{HP}(u,v) = 1 - H_{LP}(u,v)$', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Max difference between g_HP and (f - g_LP): {np.max(np.abs(g_hp - g_diff)):.2e}")
print("This confirms: highpass filtered = original - lowpass filtered")

## 3. Ideal Highpass Filter (IHPF)

The **Ideal Highpass Filter** is obtained from the Ideal Lowpass Filter:

$$H(u,v) = \begin{cases} 0 & \text{if } D(u,v) \leq D_0 \\ 1 & \text{if } D(u,v) > D_0 \end{cases}$$

Where:
- $D(u,v) = \sqrt{(u - M/2)^2 + (v - N/2)^2}$ is the distance from the center of the frequency rectangle
- $D_0$ is the cutoff frequency

### Characteristics:
- **Sharp cutoff**: All frequencies below $D_0$ are completely removed
- **Ringing artifacts**: The sharp transition causes **Gibbs phenomenon** (ringing) in the filtered image
- The IHPF completely removes the DC component ($F(0,0) = 0$), so the average intensity of the filtered image is zero
- Not used in practice due to ringing, but important for understanding

In [ ]:
# Ideal Highpass Filter
def ideal_highpass(shape, D0):
    """Create an Ideal Highpass Filter."""
    D = distance_from_center(shape)
    H = np.zeros(shape, dtype=np.float64)
    H[D > D0] = 1.0
    return H


# Show IHPF for different cutoff frequencies
D0_values = [10, 30, 60]

fig, axes = plt.subplots(3, 3, figsize=(15, 14))

for i, D0 in enumerate(D0_values):
    H = ideal_highpass(test_img.shape, D0)
    g = apply_filter(test_img, H)
    
    # Transfer function
    axes[i, 0].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f'IHPF: $D_0 = {D0}$', fontsize=12)
    axes[i, 0].axis('off')
    
    # Radial cross section
    center = test_img.shape[0] // 2
    radial = H[center, center:]
    axes[i, 1].plot(np.arange(len(radial)), radial, 'r-', linewidth=2)
    axes[i, 1].axvline(x=D0, color='gray', linestyle='--', alpha=0.7)
    axes[i, 1].set_title(f'Radial Profile ($D_0 = {D0}$)', fontsize=12)
    axes[i, 1].set_xlabel('D(u,v)')
    axes[i, 1].set_ylabel('H(u,v)')
    axes[i, 1].set_ylim([-0.05, 1.05])
    axes[i, 1].grid(True, alpha=0.3)
    
    # Filtered result
    axes[i, 2].imshow(g, cmap='gray')
    axes[i, 2].set_title(f'Filtered Result ($D_0 = {D0}$)', fontsize=12)
    axes[i, 2].axis('off')

plt.suptitle('Ideal Highpass Filter (IHPF)\nNote the ringing artifacts from the sharp cutoff',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in to show ringing artifacts clearly
H_ihpf = ideal_highpass(test_img.shape, 20)
g_ihpf = apply_filter(test_img, H_ihpf)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(g_ihpf, cmap='gray')
axes[1].set_title('IHPF Filtered ($D_0 = 20$)', fontsize=12)
axes[1].axis('off')

# Zoomed view to see ringing
y1, y2, x1, x2 = 50, 150, 50, 150
axes[2].imshow(g_ihpf[y1:y2, x1:x2], cmap='gray')
axes[2].set_title('Zoomed View: Ringing Artifacts Visible', fontsize=12)
axes[2].axis('off')

plt.suptitle('IHPF Ringing Artifacts (Gibbs Phenomenon)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The sharp cutoff in the IHPF causes oscillations (ringing) near edges.")
print("This is analogous to the ringing in the ideal lowpass filter.")

## 4. Gaussian Highpass Filter (GHPF)

The **Gaussian Highpass Filter** is obtained from the Gaussian Lowpass Filter:

$$H(u,v) = 1 - e^{-D^2(u,v) / (2D_0^2)}$$

Where:
- $D(u,v)$ is the distance from the center of the frequency rectangle
- $D_0$ is the cutoff frequency (at $H = 1 - e^{-1/2} \approx 0.3935$)

### Characteristics:
- **Smooth transition**: No sharp cutoff, hence **no ringing artifacts**
- The Gaussian is the only function whose Fourier transform is also Gaussian
- Provides clean sharpening without introducing oscillations
- The transition from 0 to 1 is gradual and symmetric about $D_0$
- Most commonly used highpass filter in practice

In [ ]:
# Gaussian Highpass Filter
def gaussian_highpass(shape, D0):
    """Create a Gaussian Highpass Filter."""
    D = distance_from_center(shape)
    H = 1 - np.exp(-D**2 / (2 * D0**2))
    return H


# Show GHPF for different cutoff frequencies
D0_values = [10, 30, 60]

fig, axes = plt.subplots(3, 3, figsize=(15, 14))

for i, D0 in enumerate(D0_values):
    H = gaussian_highpass(test_img.shape, D0)
    g = apply_filter(test_img, H)
    
    # Transfer function
    axes[i, 0].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f'GHPF: $D_0 = {D0}$', fontsize=12)
    axes[i, 0].axis('off')
    
    # Radial cross section
    center = test_img.shape[0] // 2
    radial = H[center, center:]
    axes[i, 1].plot(np.arange(len(radial)), radial, 'g-', linewidth=2)
    axes[i, 1].axvline(x=D0, color='gray', linestyle='--', alpha=0.7)
    axes[i, 1].set_title(f'Radial Profile ($D_0 = {D0}$)', fontsize=12)
    axes[i, 1].set_xlabel('D(u,v)')
    axes[i, 1].set_ylabel('H(u,v)')
    axes[i, 1].set_ylim([-0.05, 1.05])
    axes[i, 1].grid(True, alpha=0.3)
    
    # Filtered result
    axes[i, 2].imshow(g, cmap='gray')
    axes[i, 2].set_title(f'Filtered Result ($D_0 = {D0}$)', fontsize=12)
    axes[i, 2].axis('off')

plt.suptitle('Gaussian Highpass Filter (GHPF)\nSmooth transition, no ringing artifacts',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Butterworth Highpass Filter (BHPF)

The **Butterworth Highpass Filter** of order $n$ is:

$$H(u,v) = \frac{1}{1 + \left[\frac{D_0}{D(u,v)}\right]^{2n}}$$

Where:
- $D(u,v)$ is the distance from the center
- $D_0$ is the cutoff frequency (at $H = 0.5$)
- $n$ is the filter order

### Characteristics:
- **Tunable sharpness**: The order $n$ controls the transition sharpness
  - Low order ($n = 1$): smooth transition, similar to Gaussian
  - High order ($n \to \infty$): approaches the Ideal HPF
- At $D = D_0$: $H = 0.5$ (the filter value is exactly 0.5 at the cutoff)
- Provides a good balance between sharpening quality and ringing control
- For $n \leq 2$, ringing is usually negligible

In [ ]:
# Butterworth Highpass Filter
def butterworth_highpass(shape, D0, n):
    """Create a Butterworth Highpass Filter of order n."""
    D = distance_from_center(shape)
    # Avoid division by zero at the center
    D_safe = np.where(D == 0, 1e-10, D)
    H = 1 / (1 + (D0 / D_safe)**( 2 * n))
    return H


# Show BHPF for different cutoff frequencies with order n=2
D0_values = [10, 30, 60]

fig, axes = plt.subplots(3, 3, figsize=(15, 14))

for i, D0 in enumerate(D0_values):
    H = butterworth_highpass(test_img.shape, D0, n=2)
    g = apply_filter(test_img, H)
    
    # Transfer function
    axes[i, 0].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f'BHPF: $D_0 = {D0}$, $n = 2$', fontsize=12)
    axes[i, 0].axis('off')
    
    # Radial cross section
    center = test_img.shape[0] // 2
    radial = H[center, center:]
    axes[i, 1].plot(np.arange(len(radial)), radial, 'b-', linewidth=2)
    axes[i, 1].axvline(x=D0, color='gray', linestyle='--', alpha=0.7)
    axes[i, 1].axhline(y=0.5, color='orange', linestyle=':', alpha=0.7, label='H = 0.5')
    axes[i, 1].set_title(f'Radial Profile ($D_0 = {D0}$, $n = 2$)', fontsize=12)
    axes[i, 1].set_xlabel('D(u,v)')
    axes[i, 1].set_ylabel('H(u,v)')
    axes[i, 1].set_ylim([-0.05, 1.05])
    axes[i, 1].legend(fontsize=9)
    axes[i, 1].grid(True, alpha=0.3)
    
    # Filtered result
    axes[i, 2].imshow(g, cmap='gray')
    axes[i, 2].set_title(f'Filtered Result ($D_0 = {D0}$, $n = 2$)', fontsize=12)
    axes[i, 2].axis('off')

plt.suptitle('Butterworth Highpass Filter (BHPF), Order n = 2\nH = 0.5 at D = D_0',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Effect of filter order n on the Butterworth Highpass Filter
D0 = 30
orders = [1, 2, 5, 10, 20]

fig, axes = plt.subplots(2, len(orders), figsize=(18, 8))

for i, n in enumerate(orders):
    H = butterworth_highpass(test_img.shape, D0, n)
    g = apply_filter(test_img, H)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'n = {n}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(g, cmap='gray')
    axes[1, i].set_title(f'Result (n = {n})', fontsize=12)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Transfer Function', fontsize=11)
axes[1, 0].set_ylabel('Filtered Image', fontsize=11)

plt.suptitle(f'BHPF: Effect of Filter Order n (D_0 = {D0})\n'
             'Higher order approaches Ideal HPF, introducing ringing',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Radial profiles comparison for different orders
D0 = 30
D = distance_from_center(test_img.shape)
center = test_img.shape[0] // 2
freqs = np.arange(center)

plt.figure(figsize=(10, 6))

for n in [1, 2, 5, 10, 20]:
    H = butterworth_highpass(test_img.shape, D0, n)
    radial = H[center, center:]
    plt.plot(freqs, radial, linewidth=2, label=f'n = {n}')

# Also show ideal HPF for reference
H_ideal = ideal_highpass(test_img.shape, D0)
radial_ideal = H_ideal[center, center:]
plt.plot(freqs, radial_ideal, 'k--', linewidth=2, label='Ideal HPF')

plt.axvline(x=D0, color='gray', linestyle=':', alpha=0.5)
plt.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Distance from center D(u,v)', fontsize=12)
plt.ylabel('H(u,v)', fontsize=12)
plt.title(f'BHPF Radial Profiles for Different Orders ($D_0 = {D0}$)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([-0.05, 1.05])
plt.xlim([0, 80])
plt.show()

## 6. Comparison of All Three Highpass Filters

Let us compare the three highpass filters side by side:

| Filter | Formula | Transition | Ringing |
|--------|---------|------------|--------|
| **IHPF** | $H = \begin{cases} 0 & D \leq D_0 \\ 1 & D > D_0 \end{cases}$ | Sharp | Severe |
| **GHPF** | $H = 1 - e^{-D^2/(2D_0^2)}$ | Smooth | None |
| **BHPF** | $H = \frac{1}{1 + (D_0/D)^{2n}}$ | Tunable (order $n$) | Mild for low $n$ |

In [ ]:
# Side-by-side comparison of all three highpass filters
D0 = 30

H_ideal = ideal_highpass(test_img.shape, D0)
H_gauss = gaussian_highpass(test_img.shape, D0)
H_butter = butterworth_highpass(test_img.shape, D0, n=2)

g_ideal = apply_filter(test_img, H_ideal)
g_gauss = apply_filter(test_img, H_gauss)
g_butter = apply_filter(test_img, H_butter)

fig, axes = plt.subplots(3, 3, figsize=(15, 14))

filters = [(H_ideal, g_ideal, 'IHPF', 'r'),
           (H_gauss, g_gauss, 'GHPF', 'g'),
           (H_butter, g_butter, 'BHPF (n=2)', 'b')]

center = test_img.shape[0] // 2
freqs = np.arange(center)

for i, (H, g, name, color) in enumerate(filters):
    # Transfer function image
    axes[i, 0].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f'{name}: $D_0 = {D0}$', fontsize=12)
    axes[i, 0].axis('off')
    
    # Radial cross section
    radial = H[center, center:]
    axes[i, 1].plot(freqs, radial, color=color, linewidth=2)
    axes[i, 1].axvline(x=D0, color='gray', linestyle='--', alpha=0.7)
    axes[i, 1].set_title(f'{name} Radial Profile', fontsize=12)
    axes[i, 1].set_xlabel('D(u,v)')
    axes[i, 1].set_ylabel('H(u,v)')
    axes[i, 1].set_ylim([-0.05, 1.05])
    axes[i, 1].set_xlim([0, 80])
    axes[i, 1].grid(True, alpha=0.3)
    
    # Filtered result
    axes[i, 2].imshow(g, cmap='gray')
    axes[i, 2].set_title(f'{name} Result', fontsize=12)
    axes[i, 2].axis('off')

plt.suptitle(f'Comparison of Highpass Filters ($D_0 = {D0}$)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Overlay all three radial profiles on one plot
D0 = 30
center = test_img.shape[0] // 2
freqs = np.arange(center)

H_i = ideal_highpass(test_img.shape, D0)
H_g = gaussian_highpass(test_img.shape, D0)
H_b = butterworth_highpass(test_img.shape, D0, n=2)

plt.figure(figsize=(10, 6))
plt.plot(freqs, H_i[center, center:], 'r-', linewidth=2, label='Ideal HPF')
plt.plot(freqs, H_g[center, center:], 'g-', linewidth=2, label='Gaussian HPF')
plt.plot(freqs, H_b[center, center:], 'b-', linewidth=2, label='Butterworth HPF (n=2)')

plt.axvline(x=D0, color='gray', linestyle='--', alpha=0.5, label=f'$D_0 = {D0}$')
plt.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

plt.xlabel('Distance from center D(u,v)', fontsize=12)
plt.ylabel('H(u,v)', fontsize=12)
plt.title(f'Highpass Filter Radial Profiles ($D_0 = {D0}$)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([-0.05, 1.05])
plt.xlim([0, 80])
plt.show()

print("Key observations:")
print("  - IHPF has a sharp (step) transition at D0")
print("  - GHPF has the smoothest transition")
print("  - BHPF falls between IHPF and GHPF, depending on order n")

## 7. High-Frequency Emphasis (HFE) Filtering

A pure highpass filter removes the DC component entirely, resulting in an image with zero average intensity. This often produces images that appear very dark or have poor overall contrast.

**High-frequency emphasis filtering** addresses this by preserving some of the low-frequency content while boosting high frequencies:

$$\boxed{H_{hfe}(u,v) = a + b \cdot H_{HP}(u,v)}$$

Where:
- $a \geq 0$ is the **offset** (controls how much of the original low-frequency content is preserved)
- $b > a$ is the **high-frequency multiplier** (controls the emphasis on high frequencies)
- $H_{HP}(u,v)$ is any highpass filter

### How it works:
- When $a = 0, b = 1$: Standard highpass filtering (no emphasis)
- When $a > 0$: Some low-frequency content is preserved (the DC component is not zero)
- When $b > 1$: High frequencies are amplified beyond their original values

### Range of the HFE filter:
- At $D = 0$ (DC): $H_{hfe} = a + b \cdot 0 = a$ (low frequencies are scaled by $a$)
- At $D \gg D_0$: $H_{hfe} = a + b \cdot 1 = a + b$ (high frequencies are scaled by $a + b$)

Common choices: $a = 0.5$, $b = 2.0$ (preserves half the DC, triples the high frequencies relative to DC).

In [ ]:
# High-Frequency Emphasis Filtering
D0 = 30
H_hp = gaussian_highpass(test_img.shape, D0)

# Different (a, b) combinations
params = [(0, 1, 'Standard HP\n(a=0, b=1)'),
          (0.5, 1, 'Mild emphasis\n(a=0.5, b=1)'),
          (0.5, 2, 'Strong emphasis\n(a=0.5, b=2)'),
          (0.25, 3, 'Very strong emphasis\n(a=0.25, b=3)')]

fig, axes = plt.subplots(2, len(params), figsize=(18, 9))

center = test_img.shape[0] // 2
freqs = np.arange(center)

for i, (a, b, title) in enumerate(params):
    H_hfe = a + b * H_hp
    g_hfe = apply_filter(test_img, H_hfe)
    
    # Radial profile
    radial = H_hfe[center, center:]
    axes[0, i].plot(freqs, radial, 'b-', linewidth=2)
    axes[0, i].axhline(y=a, color='r', linestyle='--', alpha=0.7, label=f'a = {a}')
    axes[0, i].axhline(y=a+b, color='g', linestyle='--', alpha=0.7, label=f'a+b = {a+b}')
    axes[0, i].set_title(title, fontsize=11)
    axes[0, i].set_xlabel('D(u,v)')
    axes[0, i].set_ylabel('$H_{hfe}(u,v)$')
    axes[0, i].legend(fontsize=8)
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].set_xlim([0, 80])
    axes[0, i].set_ylim([-0.1, max(a+b, 1) + 0.5])
    
    # Filtered and rescaled result
    g_display = np.clip(g_hfe, 0, 255)
    axes[1, i].imshow(g_display, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Result (a={a}, b={b})', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('$H_{hfe}(u,v)$', fontsize=12)
axes[1, 0].set_ylabel('Filtered Image', fontsize=12)

plt.suptitle('High-Frequency Emphasis Filtering: $H_{hfe}(u,v) = a + b \\cdot H_{HP}(u,v)$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("  - Standard HP (a=0): image is dark, only edges visible")
print("  - Adding offset a > 0: preserves overall brightness")
print("  - Increasing b: stronger edge enhancement")

In [ ]:
# HFE combined with histogram equalization (common technique)
def histogram_equalize(img):
    """Simple histogram equalization for 8-bit images."""
    img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
    hist, bins = np.histogram(img_uint8.flatten(), 256, [0, 256])
    cdf = hist.cumsum()
    # Mask zero values
    cdf_m = np.ma.masked_equal(cdf, 0)
    cdf_m = (cdf_m - cdf_m.min()) * 255 / (cdf_m.max() - cdf_m.min())
    cdf_final = np.ma.filled(cdf_m, 0).astype(np.uint8)
    return cdf_final[img_uint8]


D0 = 30
a, b = 0.5, 2.0
H_hp = gaussian_highpass(test_img.shape, D0)
H_hfe = a + b * H_hp

g_hfe = apply_filter(test_img, H_hfe)
g_hfe_eq = histogram_equalize(g_hfe)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(np.clip(g_hfe, 0, 255), cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'HFE (a={a}, b={b})', fontsize=12)
axes[1].axis('off')

axes[2].imshow(g_hfe_eq, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('HFE + Histogram Equalization', fontsize=12)
axes[2].axis('off')

plt.suptitle('High-Frequency Emphasis + Histogram Equalization\n'
             'A common combination for image enhancement',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Laplacian in the Frequency Domain

The **Laplacian** operator is a second-order derivative operator used for edge detection and image sharpening. In the **spatial domain**, the Laplacian of a 2-D function $f(x,y)$ is:

$$\nabla^2 f(x,y) = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$

### Laplacian in the Frequency Domain

A key property of the Fourier transform is that differentiation in the spatial domain corresponds to multiplication by frequency variables in the frequency domain.

For the continuous case:

$$\mathcal{F}\{\nabla^2 f(x,y)\} = -(2\pi)^2 (u^2 + v^2) \cdot F(u,v)$$

Therefore, the Laplacian filter in the frequency domain has the transfer function:

$$\boxed{H(u,v) = -4\pi^2 (u^2 + v^2)}$$

For the **centered** discrete case (after shifting), this becomes:

$$H(u,v) = -4\pi^2 \left[\left(u - \frac{M}{2}\right)^2 + \left(v - \frac{N}{2}\right)^2\right]$$

### Image Sharpening with the Laplacian

The **Laplacian-sharpened** image is obtained by subtracting the Laplacian from the original:

$$g(x,y) = f(x,y) - \nabla^2 f(x,y)$$

Note: The subtraction is used because the Laplacian kernel used here has a negative center. If the center is positive, use addition instead.

In [ ]:
# Laplacian in the frequency domain
M, N = test_img.shape
u = np.arange(M) - M // 2
v = np.arange(N) - N // 2
V, U = np.meshgrid(v, u)

# Laplacian transfer function: H(u,v) = -4*pi^2*(u^2 + v^2)
H_laplacian = -4 * (np.pi ** 2) * (U**2 + V**2)

# Apply the Laplacian filter in the frequency domain
F = np.fft.fftshift(np.fft.fft2(test_img))
G_laplacian = H_laplacian * F
laplacian_result = np.real(np.fft.ifft2(np.fft.ifftshift(G_laplacian)))

# Sharpened image: f(x,y) - laplacian(f)
sharpened = test_img - laplacian_result

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original Image $f(x,y)$', fontsize=12)
axes[0, 0].axis('off')

# Display the transfer function (normalized for visualization)
H_display = H_laplacian / np.min(H_laplacian)  # normalize to [0, 1] for display
axes[0, 1].imshow(H_display, cmap='gray')
axes[0, 1].set_title('Laplacian $H(u,v) = -4\\pi^2(u^2 + v^2)$\n(normalized for display)', fontsize=12)
axes[0, 1].axis('off')

# Radial profile of the Laplacian
center = M // 2
radial_lap = H_laplacian[center, center:]
freqs = np.arange(len(radial_lap))
axes[0, 2].plot(freqs, radial_lap, 'purple', linewidth=2)
axes[0, 2].set_title('Radial Profile of $H(u,v)$', fontsize=12)
axes[0, 2].set_xlabel('D(u,v)')
axes[0, 2].set_ylabel('$H(u,v)$')
axes[0, 2].grid(True, alpha=0.3)

# Laplacian result (scaled for display)
lap_display = laplacian_result - laplacian_result.min()
lap_display = lap_display / lap_display.max() * 255
axes[1, 0].imshow(lap_display, cmap='gray')
axes[1, 0].set_title('Laplacian $\\nabla^2 f$ (scaled)', fontsize=12)
axes[1, 0].axis('off')

# Sharpened result
sharp_display = np.clip(sharpened, 0, 255)
axes[1, 1].imshow(sharp_display, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title('Sharpened: $f - \\nabla^2 f$', fontsize=12)
axes[1, 1].axis('off')

# Difference to highlight enhancement
diff = np.abs(sharpened - test_img)
diff = diff / diff.max() * 255
axes[1, 2].imshow(diff, cmap='hot')
axes[1, 2].set_title('Enhancement Map: $|g - f|$', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Laplacian in the Frequency Domain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare spatial Laplacian vs frequency domain Laplacian
from scipy.ndimage import convolve

# Spatial domain Laplacian kernel
laplacian_kernel = np.array([[0,  1, 0],
                             [1, -4, 1],
                             [0,  1, 0]], dtype=np.float64)

# Spatial Laplacian
lap_spatial = convolve(test_img, laplacian_kernel, mode='reflect')

# Frequency domain Laplacian (already computed above)
# Normalize both for comparison
lap_spatial_norm = (lap_spatial - lap_spatial.min()) / (lap_spatial.max() - lap_spatial.min())
lap_freq_norm = (laplacian_result - laplacian_result.min()) / (laplacian_result.max() - laplacian_result.min())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(lap_spatial_norm, cmap='gray')
axes[0].set_title('Spatial Domain Laplacian\n(3x3 kernel convolution)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(lap_freq_norm, cmap='gray')
axes[1].set_title('Frequency Domain Laplacian\n($H = -4\\pi^2(u^2 + v^2)$)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(np.abs(lap_spatial_norm - lap_freq_norm), cmap='hot')
axes[2].set_title('Absolute Difference', fontsize=12)
axes[2].axis('off')

plt.suptitle('Spatial vs Frequency Domain Laplacian\nBoth detect the same edges (continuous Laplacian has broader response)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The spatial 3x3 Laplacian kernel is an approximation of the continuous Laplacian.")
print("The frequency domain Laplacian applies the exact continuous operator.")
print("Small differences arise because the discrete kernel is a finite-difference approximation.")

## 9. Practical Example: Sharpening a Biomedical-Like Image

Let us apply all the techniques from this lesson to sharpen a more realistic synthetic biomedical image.

In [ ]:
# Create a blurred biomedical-like image (simulating a soft-focus scan)
def create_biomedical_image(size=256):
    """Create a synthetic biomedical image simulating a blurred tissue scan."""
    img = np.zeros((size, size), dtype=np.float64)
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    # Background tissue
    img += 80 + 15 * np.sin(2 * np.pi * 3 * X / size) * np.cos(2 * np.pi * 2 * Y / size)
    
    # Large organ boundary
    organ = ((X - cx) / 80)**2 + ((Y - cy) / 60)**2
    img[organ <= 1] += 60
    
    # Internal structures (vessels, ducts)
    for (dx, dy, r, val) in [(-30, -20, 15, 50), (25, 15, 12, -40),
                             (0, 30, 8, 70), (-20, 25, 6, -30),
                             (35, -10, 10, 45)]:
        dist = np.sqrt((X - cx - dx)**2 + (Y - cy - dy)**2)
        img[dist <= r] += val
    
    # Thin boundary lines
    ring = np.abs(organ - 1)
    img[ring < 0.03] = 200
    
    img = np.clip(img, 0, 255)
    
    # Apply Gaussian blur to simulate soft focus
    F = np.fft.fftshift(np.fft.fft2(img))
    D = distance_from_center(img.shape)
    H_blur = np.exp(-D**2 / (2 * 40**2))
    img_blurred = np.real(np.fft.ifft2(np.fft.ifftshift(H_blur * F)))
    
    return np.clip(img_blurred, 0, 255)


bio_img = create_biomedical_image(256)

# Apply different sharpening methods
D0 = 25

# Method 1: Gaussian HPF
H_ghpf = gaussian_highpass(bio_img.shape, D0)
g_ghpf = apply_filter(bio_img, H_ghpf)

# Method 2: HFE with Gaussian HPF
H_hfe = 0.5 + 2.0 * H_ghpf
g_hfe = apply_filter(bio_img, H_hfe)

# Method 3: Laplacian sharpening
M_img, N_img = bio_img.shape
u_arr = np.arange(M_img) - M_img // 2
v_arr = np.arange(N_img) - N_img // 2
VV, UU = np.meshgrid(v_arr, u_arr)
H_lap = -4 * (np.pi ** 2) * (UU**2 + VV**2)
F_bio = np.fft.fftshift(np.fft.fft2(bio_img))
lap_bio = np.real(np.fft.ifft2(np.fft.ifftshift(H_lap * F_bio)))
g_lap = bio_img - lap_bio

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(bio_img, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Blurred Biomedical Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(g_ghpf, cmap='gray')
axes[0, 1].set_title(f'GHPF ($D_0 = {D0}$)\nEdges only', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(np.clip(g_hfe, 0, 255), cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title('HFE (a=0.5, b=2.0)\nSharpened with context', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(np.clip(g_lap, 0, 255), cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Laplacian Sharpening\n$f - \\nabla^2 f$', fontsize=12)
axes[1, 0].axis('off')

# HFE + histogram equalization
g_hfe_eq = histogram_equalize(np.clip(g_hfe, 0, 255))
axes[1, 1].imshow(g_hfe_eq, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title('HFE + Hist. Equalization\nBest overall enhancement', fontsize=12)
axes[1, 1].axis('off')

# Combined: Laplacian + HFE
H_combined = 0.5 + 2.0 * gaussian_highpass(bio_img.shape, 20)
g_combined = apply_filter(bio_img, H_combined)
g_combined_eq = histogram_equalize(np.clip(g_combined, 0, 255))
axes[1, 2].imshow(g_combined_eq, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title('GHPF HFE + Hist. Eq.\n($D_0=20$, a=0.5, b=2.0)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Sharpening a Biomedical Image: Comparison of Methods',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

What we learned:

1. **Highpass filters** attenuate low frequencies and pass high frequencies, used for sharpening and edge detection
2. **Relationship**: $H_{HP}(u,v) = 1 - H_{LP}(u,v)$ -- any lowpass filter can be converted to a highpass filter
3. **Ideal Highpass Filter (IHPF)**: Sharp cutoff at $D_0$, causes ringing (Gibbs phenomenon)
4. **Gaussian Highpass Filter (GHPF)**: $H = 1 - e^{-D^2/(2D_0^2)}$, smooth transition, no ringing
5. **Butterworth Highpass Filter (BHPF)**: $H = 1/(1 + (D_0/D)^{2n})$, tunable sharpness via order $n$
6. **Filter comparison**: IHPF has sharp cutoff with ringing; GHPF is smoothest; BHPF is tunable between the two
7. **High-frequency emphasis**: $H_{hfe} = a + b \cdot H_{HP}$ preserves low-frequency content while boosting edges
8. **Laplacian in frequency domain**: $H(u,v) = -4\pi^2(u^2 + v^2)$, implements second-order differentiation

### Key Takeaway for Biomedical Imaging

In biomedical image processing, highpass filtering and sharpening are crucial for:
- Enhancing edges of organs, tissues, and lesions
- Improving visibility of fine structures (vessels, calcifications)
- Pre-processing before segmentation or feature extraction
- The GHPF or low-order BHPF combined with HFE are the most practical choices